# 02 · Per-structure physics decisions (SpinType + odd/even nspin)

How we decide spin treatment. Part A compares the &SYSTEM block aiida-qe emits for the four SpinType settings (NONE / COLLINEAR / NON_COLLINEAR / SPIN_ORBIT) on magnetic vs non-magnetic structures. Part B is the odd-electron nspin screen (odd total valence electrons -> must be spin-polarized). Backs PLAN.md §3.4 / §4.

*Consolidated aiida-spintype.ipynb,even-odd-electrons.ipynb (2026-06-16 notebook cleanup).*

---
## Part A — SpinType &SYSTEM comparison

In [1]:
# Compare PwBaseWorkChain SYSTEM parameters across the 4 SpinType settings.
# Goal: see exactly what aiida-qe puts in &SYSTEM for each spin treatment,
# on both a magnetic structure (Fe BCC) and a non-magnetic one (Si diamond).

from aiida import load_profile
from aiida.orm import StructureData, load_code
from aiida_quantumespresso.common.types import ElectronicType, SpinType
from aiida_quantumespresso.workflows.pw.base import PwBaseWorkChain
from ase.build import bulk

load_profile();

fe = StructureData(ase=bulk('Fe', 'bcc', a=2.87)).store()
si = StructureData(ase=bulk('Si', 'diamond', a=5.43)).store()

code = load_code('qe-7.2-pw@scarf')

PSEUDO_SR = 'PseudoDojo/0.4/PBEsol/SR/standard/upf'   # nspin=1/2 用
PSEUDO_FR = 'PseudoDojo/0.4/PBEsol/FR/standard/upf'   # noncolin / SOC 用

print(f'Fe PK={fe.pk}  Si PK={si.pk}')


Fe PK=34050  Si PK=34051


In [2]:
def inspect_spin(structure, spin_type):
    """Build a builder for the given spin_type, return SYSTEM dict + status."""
    pseudo = PSEUDO_FR if spin_type in (SpinType.NON_COLLINEAR, SpinType.SPIN_ORBIT) else PSEUDO_SR
    try:
        builder = PwBaseWorkChain.get_builder_from_protocol(
            code=code,
            structure=structure,
            protocol='moderate',
            electronic_type=ElectronicType.METAL,
            spin_type=spin_type,
            overrides={'pseudo_family': pseudo},
        )
        return builder.pw.parameters.get_dict().get('SYSTEM', {})
    except Exception as e:
        return {'__error__': str(e)[:120]}


In [3]:
import pandas as pd

SPIN_TYPES = [SpinType.NONE, SpinType.COLLINEAR, SpinType.NON_COLLINEAR, SpinType.SPIN_ORBIT]
SHORT_NAMES = ['NONE', 'COLLINEAR', 'NON_COLLINEAR', 'SPIN_ORBIT']


def compare(structure, label):
    print(f'\n{"="*80}\n{label}  (PK={structure.pk}, formula={structure.get_formula()})\n{"="*80}')
    results = {short: inspect_spin(structure, st) for short, st in zip(SHORT_NAMES, SPIN_TYPES)}
    all_keys = sorted(set().union(*[d.keys() for d in results.values()]))
    rows = []
    for key in all_keys:
        row = {'KEY': key}
        for short in SHORT_NAMES:
            v = results[short].get(key, '∅')
            # Truncate long values for display
            row[short] = str(v)[:35]
        rows.append(row)
    df = pd.DataFrame(rows).set_index('KEY')
    print(df.to_string())


compare(fe, 'Fe BCC (magnetic)')
compare(si, 'Si diamond (non-magnetic)')



Fe BCC (magnetic)  (PK=34050, formula=Fe)
                            NONE       COLLINEAR   NON_COLLINEAR      SPIN_ORBIT
KEY                                                                             
angle1                         ∅               ∅     {'Fe': 0.0}     {'Fe': 0.0}
angle2                         ∅               ∅     {'Fe': 0.0}     {'Fe': 0.0}
degauss                     0.02            0.02            0.02            0.02
ecutrho                    360.0           360.0           360.0           360.0
ecutwfc                     90.0            90.0            90.0            90.0
lspinorb                       ∅               ∅               ∅            True
noncolin                       ∅               ∅            True            True
nosym                      False           False           False           False
nspin                          ∅               2               4               4
occupations             smearing        smearing        smearing  

---
## Part B — odd-electron nspin screen

In [2]:
# Phase A nspin Step 1: odd-electron screen.
# Logic: if total valence electrons is odd, the system MUST be spin-polarized
#   (cannot pair up an odd number of electrons in a closed-shell calc).
# This is the strict half — magnetism check (for even-electron systems with
# magnetic elements) waits for full archive import.

from aiida.orm import Group, QueryBuilder, StructureData, load_group
from collections import Counter

mc3d = load_group('mc3d-pbesol-v2-structures')
pseudo_group = load_group('PseudoDojo/0.4/PBEsol/SR/standard/upf')


def n_valence_electrons(structure, pseudo_family) -> int:
    """Sum z_valence over all sites using pseudo_family's pseudos."""
    pseudos = pseudo_family.get_pseudos(structure=structure)
    return int(sum(pseudos[site.kind_name].z_valence for site in structure.sites))


# Scan all 33k structures
qb = QueryBuilder()
qb.append(Group, filters={'label': mc3d.label}, tag='g')
qb.append(StructureData, with_group='g')

n_odd, n_even, total = 0, 0, 0
nelec_hist = Counter()      # bucket by n_electrons for sanity
odd_examples = []

for s, in qb.iterall():
    n = n_valence_electrons(s, pseudo_group)
    nelec_hist[n // 10 * 10] += 1   # bucket every 10 electrons
    is_odd = (n % 2) == 1
    if is_odd:
        n_odd += 1
        if len(odd_examples) < 5:
            odd_examples.append((s.get_formula(), n))
    else:
        n_even += 1
    total += 1

print(f'Total structures scanned: {total}')
print(f'  odd-electron  (must nspin=2): {n_odd:6}  ({100*n_odd/total:5.1f}%)')
print(f'  even-electron (pending):      {n_even:6}  ({100*n_even/total:5.1f}%)')

print(f'\nValence electron count distribution (bucketed):')
for bucket in sorted(nelec_hist):
    bar = '█' * (nelec_hist[bucket] * 50 // max(nelec_hist.values()))
    print(f'  {bucket:3}–{bucket+9:3} : {nelec_hist[bucket]:5}  {bar}')

print(f'\nFirst 5 odd-electron examples:')
for formula, n in odd_examples:
    print(f'  {formula:20s} n_e = {n}')


Total structures scanned: 33142
  odd-electron  (must nspin=2):   3517  ( 10.6%)
  even-electron (pending):       29625  ( 89.4%)

Valence electron count distribution (bucketed):
    0–  9 :    90  ██
   10– 19 :   589  ██████████████████
   20– 29 :   891  ███████████████████████████
   30– 39 :   971  █████████████████████████████
   40– 49 :  1407  ███████████████████████████████████████████
   50– 59 :  1171  ███████████████████████████████████
   60– 69 :  1613  █████████████████████████████████████████████████
   70– 79 :  1286  ███████████████████████████████████████
   80– 89 :  1628  ██████████████████████████████████████████████████
   90– 99 :  1338  █████████████████████████████████████████
  100–109 :  1157  ███████████████████████████████████
  110–119 :  1007  ██████████████████████████████
  120–129 :  1202  ████████████████████████████████████
  130–139 :   988  ██████████████████████████████
  140–149 :   976  █████████████████████████████
  150–159 :   722  █████████